# HANAS Phase 1 Calibration Results

Phase 1 establishes pump flow rates and dose-response coefficients for pH and EC adjustment. The calibration results provide the dosing coefficients used by the controller benchmark.

## 1. Calibration Metrics

The Phase 1 analysis uses the following metrics:

| Metric | Role |
|---|---|
| Mean pump flow rate | Converts requested dose volume into pump runtime. |
| Flow-rate standard deviation and CV | Quantifies pump repeatability across trials. |
| Dose-response coefficient | Estimates the dose required per liter per unit change in pH or EC. |
| Coefficient standard deviation and CV | Quantifies calibration variability. |
| Adopted coefficient | Mean coefficient from the higher-volume calibration set used for controller dosing. |

## 2. Calibration Protocol

Each coefficient trial used a fresh water or test solution. The trial sequence was: prepare the target volume, measure the start pH or EC, apply the recorded correction dose, mix the solution, wait for the reading to stabilize, record the end pH or EC, and compute the dose-response coefficient.

For pH trials, EC was maintained within the target range so acid/base response could be evaluated without an EC disturbance. For EC trials, pH was maintained within the target range so nutrient-strength response could be evaluated without a pH disturbance.

For EC preparation at 3.5 L, the EC Up starting condition used 18 mL Solution A and 18 mL Solution B, while the EC Down starting condition used 42 mL Solution A and 42 mL Solution B. The mixing interval between Solution A and Solution B additions was 60 s. For EC preparation at 20 L, the EC Up starting condition used 85 mL Solution A and 85 mL Solution B, while the EC Down starting condition used 200 mL Solution A and 200 mL Solution B. The mixing interval between Solution A and Solution B additions was 3 min.

Sensor calibration used known standards rather than reservoir water. pH calibration used pH 7.0 and pH 4.0 buffer solutions, while EC calibration used 1,413 µS/cm and 12.88 mS/cm conductivity standard solutions. The combined reservoir testing sketch read pH, EC, and water temperature in the same 20 L reservoir and used temperature-compensated pH and EC values.


### Coefficient Calculations

| Correction | Change calculation | Coefficient |
|---|---|---|
| pH Up | End pH - Start pH | pH Up dose / (water volume x pH change) |
| pH Down | Start pH - End pH | pH Down dose / (water volume x pH change) |
| EC Up | End EC - Start EC | EC Up A dose / (water volume x EC change) |
| EC Down | Start EC - End EC | EC Down dilution volume / (water volume x EC change) |

For EC Up, Solution A and Solution B are dosed as paired nutrient components. The coefficient is reported per component so the same coefficient can be applied to both nutrient pumps.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)


def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "hanas_backend").exists() and (candidate / "research_analysis" / "requirements.txt").exists():
            return candidate
    raise FileNotFoundError("Could not locate the HANAS project root from the current notebook directory.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "research_analysis"
WORKBOOK_PATH = DATA_DIR / "raw_data" / "phase1" / "Phase 1 Raw Result.xlsx"
if not WORKBOOK_PATH.exists():
    raise FileNotFoundError(f"Could not find the Phase 1 workbook: {WORKBOOK_PATH}")

REPORT_DIR = DATA_DIR / "reports" / "phase1_figures"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="paper", font_scale=1.15)
print(f"Project root: {PROJECT_ROOT}")
print(f"Workbook: {WORKBOOK_PATH}")
print(f"Figure output directory: {REPORT_DIR}")

In [ ]:
sheets = pd.read_excel(WORKBOOK_PATH, sheet_name=None, engine="openpyxl")
for name, df in sheets.items():
    print(f"{name:20s} rows={len(df):4d} cols={len(df.columns):2d}")
    display(df.head(3))

## 3. Pump Flow Calibration

In [ ]:
pump_flow = sheets["pump_flow_rate"].copy()
pump_flow["Pump"] = pump_flow["Pump"].ffill().astype(str).str.replace("\n", " ", regex=False).str.strip()
for col in ["Trial", "Runtime (sec)", "Collected Volume (mL)", "Flow Rate (mL/min)"]:
    pump_flow[col] = pd.to_numeric(pump_flow[col], errors="coerce")

pump_summary = (
    pump_flow.groupby("Pump", as_index=False)
    .agg(
        trials=("Trial", "count"),
        runtime_sec=("Runtime (sec)", "mean"),
        mean_volume_ml=("Collected Volume (mL)", "mean"),
        sd_volume_ml=("Collected Volume (mL)", "std"),
        mean_flow_ml_min=("Flow Rate (mL/min)", "mean"),
        sd_flow_ml_min=("Flow Rate (mL/min)", "std"),
        min_flow_ml_min=("Flow Rate (mL/min)", "min"),
        max_flow_ml_min=("Flow Rate (mL/min)", "max"),
    )
)
pump_summary["cv_flow_pct"] = pump_summary["sd_flow_ml_min"].fillna(0) / pump_summary["mean_flow_ml_min"] * 100
pump_summary = pump_summary.sort_values("Pump")
display(pump_summary.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
plot_data = pump_summary.copy()
sns.barplot(data=plot_data, x="Pump", y="mean_flow_ml_min", color="#4C78A8", ax=ax)
ax.errorbar(
    x=np.arange(len(plot_data)),
    y=plot_data["mean_flow_ml_min"],
    yerr=plot_data["sd_flow_ml_min"].fillna(0),
    fmt="none",
    ecolor="black",
    capsize=3,
    linewidth=1,
)
ax.set_title("Pump Flow Rate Calibration")
ax.set_xlabel("Pump")
ax.set_ylabel("Flow rate (mL/min)")
ax.tick_params(axis="x", rotation=25)
fig.tight_layout()
fig.savefig(REPORT_DIR / "fig1_pump_flow_rates.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Dose-Response Calibration

In [ ]:
calibration = sheets["calibration_trials"].copy()
for col in [
    "water_volume_liters", "trial_number", "start_value", "end_value", "change_value",
    "dose_primary_ml", "dose_secondary_ml", "total_dose_ml", "coefficient", "average_coefficient",
]:
    calibration[col] = pd.to_numeric(calibration[col], errors="coerce")

calibration["dose_group"] = calibration["total_dose_ml"].round(3).astype(str) + " mL"
calibration["test_label"] = calibration["test_type"].astype(str)

def coefficient_unit(row):
    if row["test_type"] == "EC Up":
        return "mL/L/EC-unit per component"
    if row["test_type"] == "EC Down":
        return "mL/L/EC-unit"
    return "mL/L/pH-unit"

calibration["analysis_unit"] = calibration.apply(coefficient_unit, axis=1)

coefficient_by_group = (
    calibration.groupby(["test_type", "metric", "controller_action", "analysis_unit", "water_volume_liters", "total_dose_ml"], as_index=False)
    .agg(
        trials=("trial_number", "count"),
        start_mean=("start_value", "mean"),
        end_mean=("end_value", "mean"),
        change_mean=("change_value", "mean"),
        coefficient_mean=("coefficient", "mean"),
        coefficient_sd=("coefficient", "std"),
        coefficient_min=("coefficient", "min"),
        coefficient_max=("coefficient", "max"),
    )
)
coefficient_by_group["coefficient_cv_pct"] = coefficient_by_group["coefficient_sd"] / coefficient_by_group["coefficient_mean"] * 100
coefficient_by_group = coefficient_by_group.sort_values(["test_type", "total_dose_ml"])
display(coefficient_by_group.round(4))

In [ ]:
coefficient_precision = coefficient_by_group[[
    "test_type", "water_volume_liters", "total_dose_ml", "trials",
    "change_mean", "coefficient_mean", "coefficient_sd", "coefficient_cv_pct", "analysis_unit",
]].copy()
coefficient_precision = coefficient_precision.sort_values(["test_type", "water_volume_liters"])
display(coefficient_precision.round(4))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9.2, 6.4))
axes = axes.flatten()
for ax, test_type in zip(axes, sorted(calibration["test_type"].unique())):
    subset = calibration[calibration["test_type"] == test_type].copy()
    summary = coefficient_by_group[coefficient_by_group["test_type"] == test_type].copy()
    sns.stripplot(
        data=subset,
        x="water_volume_liters",
        y="coefficient",
        hue="dose_group",
        dodge=False,
        color="black",
        alpha=0.70,
        size=4,
        ax=ax,
    )
    ax.errorbar(
        x=np.arange(len(summary)),
        y=summary["coefficient_mean"],
        yerr=summary["coefficient_sd"].fillna(0),
        fmt="o",
        color="#4C78A8",
        capsize=4,
        linewidth=1.4,
        label="Mean +/- SD",
    )
    ax.set_title(test_type)
    ax.set_xlabel("Water volume (L)")
    ax.set_ylabel(summary["analysis_unit"].iloc[0])
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles[:1], labels[:1], title="Total dose", frameon=True, fontsize=8, title_fontsize=8)
fig.suptitle("Dose-Response Coefficients by Calibration Condition", y=1.02)
fig.tight_layout()
fig.savefig(REPORT_DIR / "fig2_dose_response_coefficients.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8.8, 4.4))
cv_plot = coefficient_precision.copy()
cv_plot["condition"] = cv_plot["water_volume_liters"].map(lambda v: f"{v:g} L")
sns.barplot(data=cv_plot, x="test_type", y="coefficient_cv_pct", hue="condition", palette="Set2", ax=ax)
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=2, fontsize=8)
ax.set_title("Coefficient Variability by Calibration Condition")
ax.set_xlabel("Calibration test")
ax.set_ylabel("Coefficient CV (%)")
ax.legend(title="Water volume", frameon=True)
fig.tight_layout()
fig.savefig(REPORT_DIR / "fig3_coefficient_variability.png", dpi=300, bbox_inches="tight")
plt.show()


## 5. Adopted Calibration Coefficients

The adopted dosing coefficients are computed from the higher-dose or higher-volume calibration set for each correction type. These operating coefficients align the calibration scale with the controller dosing conditions.

In [ ]:
idx = calibration.groupby("test_type")["total_dose_ml"].idxmax()
adopted_trials = calibration.loc[idx].sort_values("test_type")
max_dose_by_type = calibration.groupby("test_type")["total_dose_ml"].max().to_dict()
adopted_coefficients = (
    calibration[calibration.apply(lambda r: r["total_dose_ml"] == max_dose_by_type[r["test_type"]], axis=1)]
    .groupby(["test_type", "metric", "controller_action", "analysis_unit", "water_volume_liters", "total_dose_ml"], as_index=False)
    .agg(
        trials=("trial_number", "count"),
        adopted_coefficient=("coefficient", "mean"),
        coefficient_sd=("coefficient", "std"),
        coefficient_cv_pct=("coefficient", lambda s: s.std() / s.mean() * 100),
        mean_start=("start_value", "mean"),
        mean_end=("end_value", "mean"),
        mean_change=("change_value", "mean"),
    )
    .sort_values("test_type")
)
display(adopted_coefficients.round(4))

In [ ]:
print("Phase 1 Calibration Summary")
print("-" * 80)
for _, row in pump_summary.iterrows():
    print(f"{row['Pump']}: mean flow {row['mean_flow_ml_min']:.2f} mL/min, CV {row['cv_flow_pct']:.2f}% across {int(row['trials'])} trials.")
print("-" * 80)
for _, row in adopted_coefficients.iterrows():
    print(f"{row['test_type']}: adopted coefficient {row['adopted_coefficient']:.4f} {row['analysis_unit']} from {int(row['trials'])} trials at {row['total_dose_ml']:.0f} mL total dose.")

## 6. Result Interpretation

Across five trials per pump, flow-rate CVs ranged from 0% to 0.44% for both EC Up pumps, pH Down, and EC Down. The pH Up pump had the highest flow-rate CV at 3.79%. Its adopted dose-response coefficient had a CV of 8.14%.

The adopted dose-response coefficients are:

| Test | Adopted coefficient |
|---|---:|
| pH Up | 0.7981 mL/L/pH-unit |
| pH Down | 0.7057 mL/L/pH-unit |
| EC Up | 4.4538 mL/L/EC-unit per nutrient component |
| EC Down | 592.1217 mL/L/EC-unit |

These empirical coefficients are the means from five 20 L trials per correction type. They provide the calibration basis for dosing under the tested solution conditions. Changes in reservoir volume or solution composition require mixing, remeasurement, and validation of the dosing response.